# Multi-Agent Communication Protocol Study — Real API Version
## Communication Protocol Effects on Multi-Agent System Efficiency & Task Completion

**GR5293 Final Project — OpenAI gpt-4o-mini**

Pipeline: `Planning Agent → Execution Agent → Integration Agent`  
Protocols: `NL` | `MARKDOWN` | `JSON` | `SHARED_MEMORY`  
Task Domains: `MATH (GSM8K)` | `READING (SQuAD)` | `NEWS`  
Metrics: Token consumption · Latency · Task completion rate

## 0. Install & Import

In [1]:
!pip install openai statsmodels datasets -q

In [2]:
import json, time, copy, random, hashlib, os, re
from enum import Enum
from dataclasses import dataclass, field, asdict
from typing import Any, Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

from openai import OpenAI

print('Imports OK')

Imports OK


## 1. API Key Setup

In [3]:
# Option A — Colab Secrets (recommended)
try:
    from google.colab import userdata
    api_key = userdata.get('OPENAI_API_KEY')
except Exception:
    # Option B — environment variable or paste directly
    api_key = os.environ.get('OPENAI_API_KEY', 'sk-YOUR-KEY-HERE')

client = OpenAI(api_key=api_key)
MODEL  = 'gpt-4o-mini'

# Sanity check
resp = client.chat.completions.create(
    model=MODEL,
    messages=[{'role': 'user', 'content': 'Reply with just OK'}],
    max_tokens=5
)
print('API connected:', resp.choices[0].message.content.strip())

API connected: OK


## 2. Enums & Constants

In [4]:
class Protocol(str, Enum):
    NL            = 'NL'
    MARKDOWN      = 'MARKDOWN'
    JSON          = 'JSON'
    SHARED_MEMORY = 'SHARED_MEMORY'

class TaskDomain(str, Enum):
    MATH    = 'MATH'
    READING = 'READING'
    NEWS    = 'NEWS'

print('Protocols:', [p.value for p in Protocol])
print('Domains:  ', [d.value for d in TaskDomain])

Protocols: ['NL', 'MARKDOWN', 'JSON', 'SHARED_MEMORY']
Domains:   ['MATH', 'READING', 'NEWS']


## 3. Task Data Preparation

- **MATH**: GSM8K samples (grade-school math, numeric answer)
- **READING**: SQuAD samples (reading comprehension, text answer)  
- **NEWS**: Pre-prepared news articles with pre-extracted key facts

In [5]:
# ── GSM8K samples ──────────────────────────────────────────────────────────────
# We load a small subset. Each has 'question' and a numeric 'answer'.
try:
    from datasets import load_dataset
    gsm8k_full = load_dataset('openai/gsm8k', 'main', split='test')
    GSM8K_SAMPLES = []
    for item in list(gsm8k_full)[:20]:
        # GSM8K answer format: "#### <number>"
        ans_text = item['answer'].split('####')[-1].strip().replace(',', '')
        GSM8K_SAMPLES.append({
            'question': item['question'],
            'answer': ans_text
        })
    print(f'Loaded {len(GSM8K_SAMPLES)} GSM8K samples from HuggingFace')
except Exception as e:
    print(f'HuggingFace load failed ({e}), using built-in samples')
    GSM8K_SAMPLES = [
        {'question': 'Janet\'s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells every remaining egg at the farmers\' market for $2 each. How much does she make every day at the farmers\' market?',
         'answer': '18'},
        {'question': 'A robe takes 2 bolts of blue fiber and half that much white fiber. How many bolts in total does it take?',
         'answer': '3'},
        {'question': 'Josh decides to try flipping a house. He buys a house for $80,000 and then puts in $50,000 in repairs. This increased the value of the house by 150%. How much profit did he make?',
         'answer': '70000'},
        {'question': 'James decides to run 3 sprints 3 times a week. He runs 60 meters each sprint. How many total meters does he run a week?',
         'answer': '540'},
        {'question': 'Every day, Wendi feeds each of her chickens three cups of mixed chicken feed, containing seeds, mealworms and vegetables to help keep them healthy. She gives the chickens their feed in three separate meals. In the morning, she gives her flock of chickens 15 cups of feed. In the afternoon, she gives her chickens another 25 cups of feed. If the final meal of the day requires 2 cups of feed per chicken, how many chickens does Wendi have?',
         'answer': '20'},
    ]
    print(f'Using {len(GSM8K_SAMPLES)} built-in GSM8K samples')

print(f'  Sample Q: {GSM8K_SAMPLES[0]["question"][:80]}...')
print(f'  Sample A: {GSM8K_SAMPLES[0]["answer"]}')

README.md: 0.00B [00:00, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

Loaded 20 GSM8K samples from HuggingFace
  Sample Q: Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning an...
  Sample A: 18


In [6]:
# ── SQuAD samples ─────────────────────────────────────────────────────────────
# Each has 'context', 'question', and 'answers' (list of acceptable strings).
try:
    from datasets import load_dataset
    squad_full = load_dataset('rajpurkar/squad', split='validation')
    SQUAD_SAMPLES = []
    for item in list(squad_full)[:20]:
        SQUAD_SAMPLES.append({
            'context': item['context'],
            'question': item['question'],
            'answers': item['answers']['text']  # list of acceptable answers
        })
    print(f'Loaded {len(SQUAD_SAMPLES)} SQuAD samples from HuggingFace')
except Exception as e:
    print(f'HuggingFace load failed ({e}), using built-in samples')
    SQUAD_SAMPLES = [
        {'context': 'Architecturally, the school has a Catholic character. Atop the Main Building\'s gold dome is a golden statue of the Virgin Mary. The Main Building was designed by Henry Drummond. It is 187 feet tall.',
         'question': 'How tall is the Main Building?',
         'answers': ['187 feet']},
        {'context': 'The city of Denver is located in the western United States. It is the capital and most populous city of Colorado. Denver\'s population was 715,522 in 2020.',
         'question': 'What is the population of Denver?',
         'answers': ['715,522', '715522']},
        {'context': 'The Nobel Prize in Physics is awarded annually by the Royal Swedish Academy of Sciences. Albert Einstein received the prize in 1921 for his discovery of the photoelectric effect.',
         'question': 'In what year did Einstein receive the Nobel Prize in Physics?',
         'answers': ['1921']},
        {'context': 'Python was created by Guido van Rossum and first released in 1991. Python\'s design philosophy emphasizes code readability with its notable use of significant whitespace.',
         'question': 'Who created Python?',
         'answers': ['Guido van Rossum']},
        {'context': 'The Amazon River is the largest river by discharge volume of water in the world. It flows through Brazil, Peru, and Colombia. Its length is approximately 6,400 km.',
         'question': 'How long is the Amazon River?',
         'answers': ['approximately 6,400 km', '6,400 km', '6400 km']},
    ]
    print(f'Using {len(SQUAD_SAMPLES)} built-in SQuAD samples')

print(f'  Sample Q: {SQUAD_SAMPLES[0]["question"]}')
print(f'  Sample A: {SQUAD_SAMPLES[0]["answers"]}')

README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/14.5M [00:00<?, ?B/s]

plain_text/validation-00000-of-00001.par(…):   0%|          | 0.00/1.82M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/87599 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10570 [00:00<?, ? examples/s]

Loaded 20 SQuAD samples from HuggingFace
  Sample Q: Which NFL team represented the AFC at Super Bowl 50?
  Sample A: ['Denver Broncos', 'Denver Broncos', 'Denver Broncos']


In [7]:
# ── NEWS samples with pre-extracted key_facts ─────────────────────────────────
# Each article has 'title', 'content', and 'key_facts' (list of strings).
# key_facts are fixed before experiments — used as ground truth for evaluation.
NEWS_SAMPLES = [
    {
        'title': 'Fed Holds Interest Rates Steady in March 2025',
        'content': (
            'The Federal Reserve held its benchmark interest rate unchanged at 4.25%-4.50% '
            'at its March 2025 meeting. Chair Jerome Powell stated that inflation remains '
            'above the 2% target at 2.8%, and the labor market shows continued strength with '
            'unemployment at 3.9%. The Fed signaled two potential rate cuts later in 2025 if '
            'inflation data cooperates. Markets reacted positively, with the S&P 500 rising 0.5%.'
        ),
        'key_facts': [
            'Fed held rate at 4.25%-4.50%',
            'inflation at 2.8%',
            'unemployment at 3.9%',
            'two potential rate cuts in 2025',
            'S&P 500 rose 0.5%'
        ]
    },
    {
        'title': 'Apple Reports Record Q1 FY2025 Earnings',
        'content': (
            'Apple reported Q1 FY2025 revenue of $124.3 billion, up 4% year-over-year. '
            'iPhone revenue reached $69.1 billion. Services revenue hit a record $26.3 billion. '
            'EPS was $2.40, beating analyst estimates of $2.35. Gross margin improved to 46.9%. '
            'CEO Tim Cook highlighted strong growth in emerging markets, particularly India.'
        ),
        'key_facts': [
            'revenue $124.3 billion',
            'iPhone revenue $69.1 billion',
            'Services revenue $26.3 billion',
            'EPS $2.40',
            'gross margin 46.9%',
            'growth in India'
        ]
    },
    {
        'title': 'Tesla Announces New Affordable EV Model',
        'content': (
            'Tesla unveiled its new affordable electric vehicle, the Model Q, priced at $25,000. '
            'The Model Q targets mass-market adoption with a 280-mile range and 0-60 mph in 5.8 seconds. '
            'Production is slated to begin at Gigafactory Texas in Q3 2025. Tesla stock rose 8% on the '
            'announcement. Analysts project annual sales of 500,000 units by 2027.'
        ),
        'key_facts': [
            'Model Q priced at $25,000',
            '280-mile range',
            'production Q3 2025 at Gigafactory Texas',
            'stock rose 8%',
            'projected 500,000 units by 2027'
        ]
    },
    {
        'title': 'Global Chip Shortage Eases as TSMC Expands',
        'content': (
            'TSMC reported that its Arizona fab is now producing 4nm chips at 90% yield rates. '
            'Global semiconductor lead times have dropped from 26 weeks to 14 weeks. '
            'TSMC revenue grew 25% YoY to $22.5 billion in Q1 2025. The company plans $40 billion '
            'in capex for 2025. Intel and Samsung are also expanding capacity but lag behind.'
        ),
        'key_facts': [
            'Arizona fab producing 4nm chips at 90% yield',
            'lead times dropped from 26 to 14 weeks',
            'TSMC revenue $22.5 billion up 25% YoY',
            '$40 billion capex for 2025',
            'Intel and Samsung lag behind'
        ]
    },
    {
        'title': 'EU Passes Comprehensive AI Regulation Act',
        'content': (
            'The European Union officially enacted the AI Act, the world\'s first comprehensive AI law. '
            'High-risk AI systems must undergo conformity assessments. Companies face fines up to 7% of '
            'global revenue for violations. The law bans social scoring and real-time biometric surveillance. '
            'Tech companies have 24 months to comply. The regulation affects all companies serving EU citizens.'
        ),
        'key_facts': [
            'first comprehensive AI law',
            'fines up to 7% of global revenue',
            'bans social scoring',
            'bans real-time biometric surveillance',
            '24 months to comply'
        ]
    },
    {
        'title': 'SpaceX Starship Completes First Orbital Flight',
        'content': (
            'SpaceX\'s Starship completed its first successful orbital flight, spending 90 minutes in orbit '
            'before splashing down in the Indian Ocean. The Super Heavy booster was caught by the launch tower. '
            'Starship reached an altitude of 250 km. NASA confirmed Starship as the lunar lander for Artemis III. '
            'SpaceX plans monthly launches starting Q2 2025. The vehicle carried a 100-ton test payload.'
        ),
        'key_facts': [
            '90 minutes in orbit',
            'splashdown in Indian Ocean',
            'booster caught by launch tower',
            'altitude 250 km',
            'lunar lander for Artemis III',
            '100-ton test payload'
        ]
    },
    {
        'title': 'Amazon Launches Drone Delivery in 10 US Cities',
        'content': (
            'Amazon expanded its Prime Air drone delivery service to 10 major US cities, including '
            'New York, Los Angeles, and Chicago. Deliveries under 5 pounds arrive within 30 minutes. '
            'The MK30 drone has a 15-mile delivery radius and operates in light rain. FAA granted '
            'beyond-visual-line-of-sight approval. Amazon targets 500 million drone deliveries annually by 2028.'
        ),
        'key_facts': [
            '10 US cities including NY LA Chicago',
            'under 5 pounds within 30 minutes',
            'MK30 drone 15-mile radius',
            'FAA beyond-visual-line-of-sight approval',
            '500 million deliveries by 2028'
        ]
    },
    {
        'title': 'OpenAI Releases GPT-5 with Reasoning Capabilities',
        'content': (
            'OpenAI released GPT-5, claiming 40% improvement on graduate-level reasoning benchmarks. '
            'The model scores 92% on the MATH benchmark and 88% on GPQA. API pricing is $15 per million '
            'input tokens and $60 per million output tokens. GPT-5 supports 256K context window. '
            'Early users report significant improvements in code generation and scientific reasoning.'
        ),
        'key_facts': [
            '40% improvement on reasoning benchmarks',
            '92% on MATH benchmark',
            '88% on GPQA',
            '$15/1M input $60/1M output tokens',
            '256K context window'
        ]
    },
    {
        'title': 'Japan Earthquake Triggers Tsunami Warning',
        'content': (
            'A 7.4 magnitude earthquake struck off the coast of Hokkaido, Japan at 3:42 AM local time. '
            'A tsunami warning was issued for coastal areas within 300 km. Waves up to 3 meters were '
            'observed. 14 people were injured and approximately 50,000 residents were evacuated. '
            'The Shinkansen bullet train service was suspended for 6 hours. No nuclear plant damage was reported.'
        ),
        'key_facts': [
            '7.4 magnitude earthquake',
            'off coast of Hokkaido',
            'tsunami warning 300 km radius',
            'waves up to 3 meters',
            '14 injured 50000 evacuated',
            'Shinkansen suspended 6 hours',
            'no nuclear plant damage'
        ]
    },
    {
        'title': 'Global Carbon Emissions Decline for First Time',
        'content': (
            'Global CO2 emissions fell 2.1% in 2024, the first annual decline not caused by economic recession. '
            'Renewable energy now accounts for 35% of global electricity generation. Solar capacity additions '
            'reached 450 GW, a record. China led with 180 GW of new solar. Coal power generation dropped 5% '
            'globally. The IEA projects emissions could fall another 3% in 2025 if trends continue.'
        ),
        'key_facts': [
            'CO2 emissions fell 2.1% in 2024',
            'renewables 35% of electricity',
            'solar capacity 450 GW record',
            'China 180 GW new solar',
            'coal power dropped 5%',
            'IEA projects 3% further decline in 2025'
        ]
    },
]

print(f'Loaded {len(NEWS_SAMPLES)} news articles')
for i, n in enumerate(NEWS_SAMPLES):
    print(f'  [{i}] {n["title"]}  ({len(n["key_facts"])} key facts)')

Loaded 10 news articles
  [0] Fed Holds Interest Rates Steady in March 2025  (5 key facts)
  [1] Apple Reports Record Q1 FY2025 Earnings  (6 key facts)
  [2] Tesla Announces New Affordable EV Model  (5 key facts)
  [3] Global Chip Shortage Eases as TSMC Expands  (5 key facts)
  [4] EU Passes Comprehensive AI Regulation Act  (5 key facts)
  [5] SpaceX Starship Completes First Orbital Flight  (6 key facts)
  [6] Amazon Launches Drone Delivery in 10 US Cities  (5 key facts)
  [7] OpenAI Releases GPT-5 with Reasoning Capabilities  (5 key facts)
  [8] Japan Earthquake Triggers Tsunami Warning  (7 key facts)
  [9] Global Carbon Emissions Decline for First Time  (6 key facts)


## 4. Communication Logger

In [8]:
@dataclass
class Message:
    run_id:       str
    protocol:     str
    task_domain:  str
    sender:       str
    receiver:     str
    content:      Any
    token_count:  int   = 0
    latency_ms:   float = 0.0
    timestamp:    float = field(default_factory=time.time)


class CommunicationLogger:
    def __init__(self):
        self.messages: List[Message] = []

    def log(self, msg: Message):
        self.messages.append(msg)

    def summary(self) -> Dict:
        if not self.messages:
            return {}
        return {
            'total_messages':   len(self.messages),
            'total_tokens':     sum(m.token_count  for m in self.messages),
            'total_latency_ms': round(sum(m.latency_ms for m in self.messages), 2),
        }

    def clear(self):
        self.messages = []

print('Logger OK')

Logger OK


## 5. LLM Wrapper

In [9]:
# ── System prompts for each agent role ────────────────────────────────────────
SYSTEM_PROMPTS = {
    'planner': (
        'You are a Planning Agent. Decompose the given task into 3-5 clear subtasks. '
        'Output ONLY the subtask list, no extra commentary.'
    ),
    'executor': (
        'You are an Execution Agent. Complete each subtask precisely based on the provided information. '
        'Output ONLY your results, no extra commentary.'
    ),
    'integrator': (
        'You are an Integration Agent. Synthesize all subtask results into a single coherent final answer. '
        'Output ONLY the final answer, no extra commentary.'
    ),
}

# ── Protocol format instructions appended to prompts ─────────────────────────
PROTOCOL_INSTRUCTIONS = {
    Protocol.NL: '',
    Protocol.MARKDOWN: (
        '\n\nFormat your response using Markdown with clear headings (##), '
        'bullet points (-), and numbered lists where appropriate.'
    ),
    Protocol.JSON: (
        '\n\nFormat your response as a valid JSON object with descriptive field names. '
        'Output ONLY the JSON, no markdown fences or extra text.'
    ),
    Protocol.SHARED_MEMORY: '',  # Shared memory uses state object, no format suffix
}


def llm_call(role: str, prompt: str, protocol: Protocol,
             temperature: float = 0.3, max_tokens: int = 150) -> Tuple[str, float, int]:
    """
    Returns (response_text, latency_ms, total_tokens).
    """
    full_prompt = prompt + PROTOCOL_INSTRUCTIONS[protocol]
    t0 = time.time()
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[
            {'role': 'system', 'content': SYSTEM_PROMPTS[role]},
            {'role': 'user',   'content': full_prompt},
        ],
        temperature=temperature,
        max_tokens=max_tokens,
    )
    latency_ms   = round((time.time() - t0) * 1000, 2)
    text         = resp.choices[0].message.content.strip()
    total_tokens = resp.usage.total_tokens
    return text, latency_ms, total_tokens


print('LLM wrapper OK')

LLM wrapper OK


## 6. Shared Memory

In [10]:
class SharedMemory:
    def __init__(self):
        self._state: Dict = {}

    def write(self, agent: str, key: str, value: Any):
        self._state[key] = value

    def read(self, key: str, default=None) -> Any:
        return self._state.get(key, default)

    def snapshot(self) -> str:
        """Return current state as string for context injection."""
        return json.dumps(self._state, default=str, ensure_ascii=False)

    def overhead_tokens(self) -> int:
        """Approximate extra tokens from holding shared state."""
        return max(1, len(self.snapshot()) // 4)

    def clear(self):
        self._state = {}

print('SharedMemory OK')

SharedMemory OK


## 7. Task Prompt Builders

Build the task prompt for each domain.  
All task content is embedded directly in the prompt — no external tool calls.

In [11]:
def build_math_prompt(sample: dict) -> str:
    return (
        f'Solve this math problem step by step and give the final numeric answer.\n\n'
        f'Problem: {sample["question"]}'
    )


def build_reading_prompt(sample: dict) -> str:
    return (
        f'Read the following passage and answer the question. '
        f'Give a short, precise answer.\n\n'
        f'Passage: {sample["context"]}\n\n'
        f'Question: {sample["question"]}'
    )


def build_news_prompt(sample: dict) -> str:
    return (
        f'Analyze the following news article. Extract and summarize all key facts, '
        f'figures, and important details.\n\n'
        f'Title: {sample["title"]}\n\n'
        f'Article: {sample["content"]}'
    )


TASK_BUILDERS = {
    TaskDomain.MATH:    build_math_prompt,
    TaskDomain.READING: build_reading_prompt,
    TaskDomain.NEWS:    build_news_prompt,
}

print('Task builders OK')

Task builders OK


## 8. Answer Evaluation

- **Math**: Extract numeric answer, exact match  
- **Reading**: Substring / fuzzy match against acceptable answers  
- **News**: Key-fact coverage ratio against pre-extracted facts

In [12]:
def extract_number(text: str) -> Optional[str]:
    """Extract the last number from text (handles commas, decimals, negatives)."""
    # Look for #### pattern first (GSM8K style)
    m = re.search(r'####\s*([\-\d,\.]+)', text)
    if m:
        return m.group(1).replace(',', '')
    # Otherwise find all numbers and take the last one
    numbers = re.findall(r'[\-]?[\d,]+\.?\d*', text)
    if numbers:
        return numbers[-1].replace(',', '')
    return None


def evaluate_math(response: str, gold_answer: str) -> float:
    """Returns 1.0 if correct, 0.0 if wrong."""
    extracted = extract_number(response)
    if extracted is None:
        return 0.0
    try:
        return 1.0 if abs(float(extracted) - float(gold_answer)) < 0.01 else 0.0
    except ValueError:
        return 1.0 if extracted.strip() == gold_answer.strip() else 0.0


def evaluate_reading(response: str, gold_answers: List[str]) -> float:
    """Returns 1.0 if any gold answer appears in response (case-insensitive)."""
    resp_lower = response.lower()
    for ans in gold_answers:
        if ans.lower() in resp_lower:
            return 1.0
    return 0.0


def evaluate_news(response: str, key_facts: List[str]) -> float:
    """Returns fraction of key facts covered in the response."""
    resp_lower = response.lower()
    covered = 0
    for fact in key_facts:
        # Check if key numbers/terms from the fact appear
        tokens = re.findall(r'[\w\.\%\$]+', fact.lower())
        # Require at least half of the tokens to match
        matches = sum(1 for t in tokens if t in resp_lower)
        if matches >= max(1, len(tokens) * 0.5):
            covered += 1
    return round(covered / len(key_facts), 3) if key_facts else 0.0


EVALUATORS = {
    TaskDomain.MATH:    lambda resp, sample: evaluate_math(resp, sample['answer']),
    TaskDomain.READING: lambda resp, sample: evaluate_reading(resp, sample['answers']),
    TaskDomain.NEWS:    lambda resp, sample: evaluate_news(resp, sample['key_facts']),
}

# Quick self-test
assert evaluate_math('The answer is 18 dollars.', '18') == 1.0
assert evaluate_math('The answer is 20.', '18') == 0.0
assert evaluate_reading('The building is 187 feet tall.', ['187 feet']) == 1.0
assert evaluate_news(
    'Revenue was $124.3 billion, iPhone at $69.1 billion, EPS $2.40',
    ['revenue $124.3 billion', 'iPhone revenue $69.1 billion', 'EPS $2.40', 'growth in India']
) == 0.75
print('Evaluators OK — all self-tests passed')

Evaluators OK — all self-tests passed


## 9. Agent Functions

Three agents with fixed roles:
1. **Planning Agent**: Decomposes task into 3-5 subtasks
2. **Execution Agent**: Completes subtasks using provided info
3. **Integration Agent**: Synthesizes results into final answer

In [13]:
def _to_str(data: Any) -> str:
    return json.dumps(data, default=str, ensure_ascii=False) if isinstance(data, dict) else str(data)


# ── Agent: Planner ────────────────────────────────────────────────────────────
def agent_planner(task_prompt: str, protocol: Protocol, domain: TaskDomain,
                  logger: CommunicationLogger, run_id: str) -> Tuple[str, int]:
    prompt = f'Decompose this task into 3-5 subtasks:\n\n{task_prompt}'
    text, ms, tokens = llm_call('planner', prompt, protocol)
    logger.log(Message(
        run_id=run_id, protocol=protocol.value, task_domain=domain.value,
        sender='Planner', receiver='Executor',
        content=text, token_count=tokens, latency_ms=ms
    ))
    return text, tokens


# ── Agent: Executor ───────────────────────────────────────────────────────────
def agent_executor(plan: str, task_prompt: str, protocol: Protocol, domain: TaskDomain,
                   logger: CommunicationLogger, run_id: str) -> Tuple[str, int]:
    prompt = (
        f'Execute the following subtasks based on the information provided.\n\n'
        f'Plan:\n{plan}\n\n'
        f'Task Information:\n{task_prompt}'
    )
    text, ms, tokens = llm_call('executor', prompt, protocol)
    logger.log(Message(
        run_id=run_id, protocol=protocol.value, task_domain=domain.value,
        sender='Executor', receiver='Integrator',
        content=text, token_count=tokens, latency_ms=ms
    ))
    return text, tokens


# ── Agent: Integrator ─────────────────────────────────────────────────────────
def agent_integrator(execution_result: str, protocol: Protocol, domain: TaskDomain,
                     logger: CommunicationLogger, run_id: str) -> Tuple[str, int]:
    if domain == TaskDomain.MATH:
        extra = ' Give the final numeric answer clearly at the end in the format: #### <number>'
    elif domain == TaskDomain.READING:
        extra = ' Give the final answer as a short, direct response.'
    else:
        extra = ' Summarize all key facts and figures from the analysis.'

    prompt = (
        f'Synthesize the following execution results into a final coherent answer.{extra}\n\n'
        f'Execution Results:\n{execution_result}'
    )
    text, ms, tokens = llm_call('integrator', prompt, protocol)
    logger.log(Message(
        run_id=run_id, protocol=protocol.value, task_domain=domain.value,
        sender='Integrator', receiver='Output',
        content=text, token_count=tokens, latency_ms=ms
    ))
    return text, tokens


print('All agents OK')

All agents OK


## 10. Pipeline Runner

In [14]:
@dataclass
class RunResult:
    run_id:           str
    protocol:         str
    task_domain:      str
    sample_index:     int
    total_tokens:     int
    total_latency_ms: float
    total_messages:   int
    completion_score: float    # 0-1: exact match or fact coverage
    final_answer:     str      # truncated


def run_pipeline(protocol: Protocol, domain: TaskDomain,
                 sample: dict, sample_idx: int, seed: int = 0) -> RunResult:

    run_id = hashlib.md5(f'{protocol}{domain}{sample_idx}{seed}'.encode()).hexdigest()[:8]
    logger = CommunicationLogger()
    shared = SharedMemory()

    # Build task prompt
    task_prompt = TASK_BUILDERS[domain](sample)

    if protocol == Protocol.SHARED_MEMORY:
        # All agents read/write to shared state
        shared.write('System', 'task_prompt', task_prompt)

        plan, _  = agent_planner(shared.read('task_prompt'), protocol, domain, logger, run_id)
        shared.write('Planner', 'plan', plan)

        exec_out, _ = agent_executor(shared.read('plan'), shared.read('task_prompt'),
                                      protocol, domain, logger, run_id)
        shared.write('Executor', 'execution_result', exec_out)

        final, _ = agent_integrator(shared.read('execution_result'), protocol, domain, logger, run_id)
        shared.write('Integrator', 'final_answer', final)

        extra_tokens = shared.overhead_tokens()
    else:
        # Sequential message passing
        plan, _     = agent_planner(task_prompt, protocol, domain, logger, run_id)
        exec_out, _ = agent_executor(plan, task_prompt, protocol, domain, logger, run_id)
        final, _    = agent_integrator(exec_out, protocol, domain, logger, run_id)
        extra_tokens = 0

    # Evaluate
    score = EVALUATORS[domain](final, sample)

    s = logger.summary()
    return RunResult(
        run_id=run_id,
        protocol=protocol.value,
        task_domain=domain.value,
        sample_index=sample_idx,
        total_tokens=s.get('total_tokens', 0) + extra_tokens,
        total_latency_ms=s.get('total_latency_ms', 0),
        total_messages=s.get('total_messages', 0),
        completion_score=score,
        final_answer=final[:200],
    )


print('Pipeline runner OK')

Pipeline runner OK


## 11. Cost Estimator

Run this **before** the full experiment to check estimated API cost.

In [15]:
# How many samples per domain
N_MATH    = min(10, len(GSM8K_SAMPLES))
N_READING = min(10, len(SQUAD_SAMPLES))
N_NEWS    = min(10, len(NEWS_SAMPLES))
N_REPS    = 1  # repetitions per sample (set >1 for variance estimation)

N_SAMPLES_TOTAL = (N_MATH + N_READING + N_NEWS) * N_REPS
N_PROTOCOLS     = len(list(Protocol))
N_TOTAL_RUNS    = N_PROTOCOLS * N_SAMPLES_TOTAL

# Each run = 3 agent calls; ~300 input + 150 output tokens per call
est_in  = N_TOTAL_RUNS * 3 * 300
est_out = N_TOTAL_RUNS * 3 * 150
est_usd = est_in / 1e6 * 0.15 + est_out / 1e6 * 0.60

print(f'Samples:       Math={N_MATH}, Reading={N_READING}, News={N_NEWS}')
print(f'Protocols:     {N_PROTOCOLS} ({", ".join(p.value for p in Protocol)})')
print(f'Reps/sample:   {N_REPS}')
print(f'Total runs:    {N_TOTAL_RUNS}')
print(f'Est. tokens:   {est_in+est_out:,}  (input {est_in:,} + output {est_out:,})')
print(f'Est. cost:     ~${est_usd:.3f} USD')

Samples:       Math=10, Reading=10, News=10
Protocols:     4 (NL, MARKDOWN, JSON, SHARED_MEMORY)
Reps/sample:   1
Total runs:    120
Est. tokens:   162,000  (input 108,000 + output 54,000)
Est. cost:     ~$0.049 USD


## 12. Run Full Experiment Grid

> ⚠️ This cell makes real API calls. Check estimated cost above first.

4 protocols × 3 domains = 12 experimental conditions

In [16]:
DOMAIN_SAMPLES = {
    TaskDomain.MATH:    GSM8K_SAMPLES[:N_MATH],
    TaskDomain.READING: SQUAD_SAMPLES[:N_READING],
    TaskDomain.NEWS:    NEWS_SAMPLES[:N_NEWS],
}

results = []
total   = N_TOTAL_RUNS
done    = 0

for protocol in Protocol:
    for domain in TaskDomain:
        samples = DOMAIN_SAMPLES[domain]
        for idx, sample in enumerate(samples):
            for rep in range(N_REPS):
                try:
                    r = run_pipeline(protocol, domain, sample, idx, seed=rep)
                    results.append(asdict(r))
                except Exception as e:
                    print(f'  ERROR {protocol.value}/{domain.value}/sample{idx}/rep{rep}: {e}')
                done += 1
                if done % 20 == 0:
                    print(f'  {done}/{total} runs done...')
                time.sleep(0.3)  # rate limit buffer

df = pd.DataFrame(results)
print(f'\nDone: {len(df)} runs | total tokens: {df["total_tokens"].sum():,}')
df.head()

  20/120 runs done...
  40/120 runs done...
  60/120 runs done...
  80/120 runs done...
  100/120 runs done...
  120/120 runs done...

Done: 120 runs | total tokens: 121,575


,run_id,protocol,task_domain,sample_index,total_tokens,total_latency_ms,total_messages,completion_score,final_answer
0,94988703,NL,MATH,0,698,7337.37,3,1.0,#### 18
1,80a938f9,NL,MATH,1,540,3695.68,3,1.0,#### 3
2,c7a4d7bd,NL,MATH,2,747,6770.84,3,1.0,"Total Cost} = 200,000 - 130,000 = 70,000\n\]\n..."
3,4f44b006,NL,MATH,3,718,5388.37,3,1.0,#### 540
4,a5803556,NL,MATH,4,919,4971.54,3,1.0,to meet the daily requirement = 60 - 40 = 20 c...


## 13. Save Raw Results

In [17]:
df.to_csv('results_raw.csv', index=False)
print(f'Saved: results_raw.csv  ({len(df)} rows)')

Saved: results_raw.csv  (120 rows)


## 14. RQ1 — Protocol Efficiency by Domain

Token consumption and latency across 4 protocols × 3 domains.

In [18]:
eff = df.groupby(['protocol', 'task_domain']).agg(
    mean_tokens  = ('total_tokens',     'mean'),
    std_tokens   = ('total_tokens',      'std'),
    mean_latency = ('total_latency_ms', 'mean'),
    std_latency  = ('total_latency_ms',  'std'),
    n_runs       = ('run_id',           'count'),
).round(2)

print('RQ1: Protocol Efficiency by Domain')
print(eff.to_string())

# ANOVA on tokens — across protocols (collapsing domains)
print('\n--- ANOVA: Token Usage by Protocol ---')
groups_tok = [df[df['protocol'] == p.value]['total_tokens'].values for p in Protocol]
f_tok, p_tok = stats.f_oneway(*groups_tok)
print(f'One-way ANOVA (tokens): F={f_tok:.3f}, p={p_tok:.4f}',
      '→ Significant' if p_tok < 0.05 else '→ Not significant')

# ANOVA on latency
groups_lat = [df[df['protocol'] == p.value]['total_latency_ms'].values for p in Protocol]
f_lat, p_lat = stats.f_oneway(*groups_lat)
print(f'One-way ANOVA (latency): F={f_lat:.3f}, p={p_lat:.4f}',
      '→ Significant' if p_lat < 0.05 else '→ Not significant')

RQ1: Protocol Efficiency by Domain
                           mean_tokens  std_tokens  mean_latency  std_latency  n_runs
protocol      task_domain                                                            
JSON          MATH               831.0      113.56       5083.49       896.96      10
              NEWS              1146.7       66.64       7334.31      1407.97      10
              READING           1003.1      136.96       4379.31      1306.42      10
MARKDOWN      MATH              1004.9      181.63       7338.56      1818.18      10
              NEWS              1189.1       28.51       7267.66       765.48      10
              READING            973.3       81.16       4414.67      1044.42      10
NL            MATH               770.3      141.86       5390.34      1396.49      10
              NEWS               937.5       79.86       7100.94      1640.39      10
              READING            718.4       26.88       2818.55       769.93      10
SHARED_MEMORY MATH 

## 15. RQ2 — Task Completion Rate by Protocol × Domain

The core research question: which protocol works best for which domain?

In [19]:
completion = df.groupby(['protocol', 'task_domain']).agg(
    mean_score = ('completion_score', 'mean'),
    std_score  = ('completion_score',  'std'),
    n_runs     = ('run_id',           'count'),
).round(3)

print('RQ2: Task Completion Rate — Protocol × Domain')
print(completion.to_string())

# Two-way ANOVA: completion_score ~ protocol + domain + interaction
try:
    import statsmodels.api as sm
    from statsmodels.formula.api import ols
    model = ols('completion_score ~ C(protocol) + C(task_domain) + C(protocol):C(task_domain)',
                data=df).fit()
    print('\nTwo-way ANOVA:')
    print(sm.stats.anova_lm(model, typ=2).round(4))
except Exception as e:
    print(f'Two-way ANOVA error: {e}')

# Per-domain ANOVA
for dom in TaskDomain:
    dom_df = df[df['task_domain'] == dom.value]
    groups = [dom_df[dom_df['protocol'] == p.value]['completion_score'].values for p in Protocol]
    f, p = stats.f_oneway(*groups)
    print(f'\n  [{dom.value}] ANOVA: F={f:.3f}, p={p:.4f}',
          '→ Significant' if p < 0.05 else '→ ns')

RQ2: Task Completion Rate — Protocol × Domain
                           mean_score  std_score  n_runs
protocol      task_domain                               
JSON          MATH              0.600      0.516      10
              NEWS              0.741      0.130      10
              READING           1.000      0.000      10
MARKDOWN      MATH              0.300      0.483      10
              NEWS              0.980      0.063      10
              READING           1.000      0.000      10
NL            MATH              0.800      0.422      10
              NEWS              0.916      0.119      10
              READING           1.000      0.000      10
SHARED_MEMORY MATH              0.800      0.422      10
              NEWS              0.952      0.077      10
              READING           1.000      0.000      10

Two-way ANOVA:
                            sum_sq     df        F  PR(>F)
C(protocol)                 0.6058    3.0   2.7026  0.0491
C(task_domain)        

## 16. Bootstrap 95% Confidence Intervals

In [20]:
def bootstrap_ci(data: np.ndarray, n_boot=2000, alpha=0.05) -> Tuple[float, float]:
    boot = [np.mean(np.random.choice(data, size=len(data), replace=True)) for _ in range(n_boot)]
    return round(np.percentile(boot, 100*alpha/2), 3), round(np.percentile(boot, 100*(1-alpha/2)), 3)

print('=== Token Usage 95% CI ===')
for p in Protocol:
    d = df[df['protocol'] == p.value]['total_tokens'].values
    lo, hi = bootstrap_ci(d)
    print(f'  {p.value:<14} mean={np.mean(d):7.1f}  95% CI [{lo}, {hi}]')

print('\n=== Completion Score 95% CI (by domain) ===')
for dom in TaskDomain:
    print(f'\n  [{dom.value}]')
    for p in Protocol:
        d = df[(df['protocol'] == p.value) & (df['task_domain'] == dom.value)]['completion_score'].values
        if len(d) > 1:
            lo, hi = bootstrap_ci(d)
            print(f'    {p.value:<14} mean={np.mean(d):.3f}  95% CI [{lo}, {hi}]')
        else:
            print(f'    {p.value:<14} mean={np.mean(d):.3f}  (too few samples for CI)')

=== Token Usage 95% CI ===
  NL             mean=  808.7  95% CI [759.766, 856.771]
  MARKDOWN       mean= 1055.8  95% CI [1002.063, 1107.582]
  JSON           mean=  993.6  95% CI [929.929, 1050.806]
  SHARED_MEMORY  mean= 1194.4  95% CI [1118.962, 1270.315]

=== Completion Score 95% CI (by domain) ===

  [MATH]
    NL             mean=0.800  95% CI [0.5, 1.0]
    MARKDOWN       mean=0.300  95% CI [0.0, 0.6]
    JSON           mean=0.600  95% CI [0.3, 0.9]
    SHARED_MEMORY  mean=0.800  95% CI [0.5, 1.0]

  [READING]
    NL             mean=1.000  95% CI [1.0, 1.0]
    MARKDOWN       mean=1.000  95% CI [1.0, 1.0]
    JSON           mean=1.000  95% CI [1.0, 1.0]
    SHARED_MEMORY  mean=1.000  95% CI [1.0, 1.0]

  [NEWS]
    NL             mean=0.916  95% CI [0.84, 0.983]
    MARKDOWN       mean=0.980  95% CI [0.94, 1.0]
    JSON           mean=0.741  95% CI [0.668, 0.823]
    SHARED_MEMORY  mean=0.952  95% CI [0.905, 1.0]


## 17. Final Summary Table

In [21]:
rows = []
for p in Protocol:
    for dom in TaskDomain:
        sub = df[(df['protocol'] == p.value) & (df['task_domain'] == dom.value)]
        rows.append({
            'Protocol':         p.value,
            'Domain':           dom.value,
            'Mean Tokens':      round(sub['total_tokens'].mean(), 1),
            'Mean Latency (ms)': round(sub['total_latency_ms'].mean(), 1),
            'Completion Rate':  round(sub['completion_score'].mean(), 3),
            'N':                len(sub),
        })

summary = pd.DataFrame(rows)
print('FINAL SUMMARY TABLE')
print(summary.to_string(index=False))
summary.to_csv('results_summary.csv', index=False)
print('\nSaved: results_summary.csv')

# Best protocol per domain
print('\n=== Best Protocol per Domain ===')
for dom in TaskDomain:
    dom_rows = summary[summary['Domain'] == dom.value]
    # Best by completion rate (primary), then by tokens (secondary)
    best = dom_rows.sort_values(['Completion Rate', 'Mean Tokens'],
                                 ascending=[False, True]).iloc[0]
    print(f'  {dom.value:<10} → {best["Protocol"]:<14} '
          f'(score={best["Completion Rate"]:.3f}, tokens={best["Mean Tokens"]:.0f})')

FINAL SUMMARY TABLE
     Protocol  Domain  Mean Tokens  Mean Latency (ms)  Completion Rate  N
           NL    MATH        770.3             5390.3            0.800 10
           NL READING        718.4             2818.6            1.000 10
           NL    NEWS        937.5             7100.9            0.916 10
     MARKDOWN    MATH       1004.9             7338.6            0.300 10
     MARKDOWN READING        973.3             4414.7            1.000 10
     MARKDOWN    NEWS       1189.1             7267.7            0.980 10
         JSON    MATH        831.0             5083.5            0.600 10
         JSON READING       1003.1             4379.3            1.000 10
         JSON    NEWS       1146.7             7334.3            0.741 10
SHARED_MEMORY    MATH       1100.4             5846.6            0.800 10
SHARED_MEMORY READING       1060.8             2418.4            1.000 10
SHARED_MEMORY    NEWS       1422.0             5330.4            0.952 10

Saved: results_su

## 18. Sample Outputs
Inspect actual LLM-generated answers per condition.

In [22]:
print('Sample outputs (first sample per condition)\n')
for p in Protocol:
    for dom in TaskDomain:
        row = df[(df['protocol'] == p.value) & (df['task_domain'] == dom.value)]
        if len(row) > 0:
            r = row.iloc[0]
            print(f'[{p.value:<14} / {dom.value:<8}]  score={r["completion_score"]:.3f}  '
                  f'tokens={r["total_tokens"]}')
            print(f'  {r["final_answer"][:120]}')
            print()

Sample outputs (first sample per condition)

[NL             / MATH    ]  score=1.000  tokens=698
  #### 18

[NL             / READING ]  score=1.000  tokens=714
  Denver Broncos.

[NL             / NEWS    ]  score=1.000  tokens=910
  In March 2025, the Federal Reserve maintained its benchmark interest rate at 4.25%-4.50%. The current inflation rate sta

[MARKDOWN       / MATH    ]  score=0.000  tokens=1115
  ## Final Calculation of Janet's Egg Situation

1. **Total Eggs Laid:**
   - Janet lays **16 eggs** per day.

2. **Total 

[MARKDOWN       / READING ]  score=1.000  tokens=1010
  ## Final Answer

- The NFL team that represented the AFC at Super Bowl 50 is the **Denver Broncos**.

[MARKDOWN       / NEWS    ]  score=1.000  tokens=1219
  ## Federal Reserve's Interest Rate Decision Summary

### Key Information
- **Main Subject**: Federal Reserve's interest 

[JSON           / MATH    ]  score=0.000  tokens=850
  {
  "total_eggs_after_laying": 16,
  "eggs_consumed_for_breakfast": 3,
  

## 19. Export Experiment Config for Reproducibility

Save all experiment parameters for the proposal and GitHub repo.

In [23]:
config = {
    'project': 'Communication Protocol Effects on Multi-Agent System Efficiency',
    'course': 'GR5293 Generative AI using Large Language Models',
    'model': MODEL,
    'max_tokens': 150,
    'temperature': 0.3,
    'protocols': [p.value for p in Protocol],
    'domains': [d.value for d in TaskDomain],
    'n_math_samples': N_MATH,
    'n_reading_samples': N_READING,
    'n_news_samples': N_NEWS,
    'n_reps': N_REPS,
    'total_runs': len(df),
    'agent_architecture': {
        'agents': ['Planning Agent', 'Execution Agent', 'Integration Agent'],
        'pipeline': 'Sequential: Planner → Executor → Integrator',
    },
    'evaluation': {
        'math': 'Numeric exact match',
        'reading': 'Substring match against gold answers',
        'news': 'Key-fact coverage ratio (pre-extracted facts)',
    },
    'metrics': ['total_tokens', 'total_latency_ms', 'completion_score'],
}

with open('experiment_config.json', 'w') as f:
    json.dump(config, f, indent=2)

print('Saved: experiment_config.json')
print(json.dumps(config, indent=2))

Saved: experiment_config.json
{
  "project": "Communication Protocol Effects on Multi-Agent System Efficiency",
  "course": "GR5293 Generative AI using Large Language Models",
  "model": "gpt-4o-mini",
  "max_tokens": 150,
  "temperature": 0.3,
  "protocols": [
    "NL",
    "MARKDOWN",
    "JSON",
    "SHARED_MEMORY"
  ],
  "domains": [
    "MATH",
    "READING",
    "NEWS"
  ],
  "n_math_samples": 10,
  "n_reading_samples": 10,
  "n_news_samples": 10,
  "n_reps": 1,
  "total_runs": 120,
  "agent_architecture": {
    "agents": [
      "Planning Agent",
      "Execution Agent",
      "Integration Agent"
    ],
    "pipeline": "Sequential: Planner \u2192 Executor \u2192 Integrator"
  },
  "evaluation": {
    "math": "Numeric exact match",
    "reading": "Substring match against gold answers",
    "news": "Key-fact coverage ratio (pre-extracted facts)"
  },
  "metrics": [
    "total_tokens",
    "total_latency_ms",
    "completion_score"
  ]
}
